# Обзор Bash для практикующих разработчиков

Этот ноутбук — компактный конспект по Bash:

- синтаксис языка (переменные, условия, циклы, функции)
- работа с файлами и процессами
- обзор ключевых утилит (`grep`, `awk`, `sed`, `find`, `xargs`, `jq`, `parallel` и др.)
- приёмы для анализа логов и автоматизации
- профессиональные флаги и best practices


## 1. Базовый синтаксис Bash

### 1.1. Переменные и подстановки


In [ ]:
# присваивание (без пробелов вокруг '=')
x=10
name="Alice"

# использование
echo "$x"
echo "Hello, $name"

# массивы
arr=(a b c)
echo "${arr[0]}"      # первый элемент
echo "${arr[@]}"      # все элементы

# ассоциативные массивы (Bash 4+)
declare -A map
map[answer]=42
echo "${map[answer]}"


### 1.2. Условия

Рекомендуется использовать `[[ ... ]]` вместо `[ ... ]`:

- корректнее работает с пробелами и regex
- меньше сюрпризов при сравнении строк


In [ ]:
x=5
file="data.txt"

if [[ $x -gt 3 && -f $file ]]; then
  echo "x > 3 и файл существует"
elif [[ $x -eq 3 ]]; then
  echo "x == 3"
else
  echo "другой случай"
fi

# популярные проверки:
# строки:   -z -n == !=
# числа:    -eq -ne -lt -gt -le -ge
# файлы:    -f -d -e -s -r -w -x


### 1.3. Циклы


In [ ]:
# простой for по диапазону
for i in {1..5}; do
  echo "i = $i"
done

# по списку файлов
for file in *.log; do
  echo "обрабатываю: $file"
done

# while + чтение файла
while IFS= read -r line; do
  echo "строка: $line"
done < input.txt


### 1.4. Функции


In [ ]:
myfunc() {
  local arg1=$1
  echo "arg1 = $arg1"
}

myfunc "hello"


### 1.5. Арифметика


In [ ]:
x=10

(( x = x + 1 ))
echo "$x"

echo $(( x * 2 ))

# альтернативно:
let "x += 5"
echo "$x"


### 1.6. Командные подстановки и расширения

Командная подстановка:


In [ ]:
today=$(date +%F)
files_count=$(ls | wc -l)

echo "Сегодня: $today"
echo "Файлов в каталоге: $files_count"


Расширения параметров (parameter expansion) — очень мощный инструмент:


In [ ]:
var="hello_world.txt"

echo "${var:-default}"   # значение или default, если пусто
echo "${#var}"           # длина строки
echo "${var/world/USER}" # замена первого вхождения
echo "${var:0:5}"        # подстрока "hello"


## 2. Работа с файловой системой и процессами

### 2.1. Навигация и файлы


In [ ]:
pwd              # текущий каталог
ls -lhS          # подробный список, сортировка по размеру
stat file.txt    # подробная информация о файле

mkdir -p dir/subdir
cp src.txt dst.txt
mv old.txt new.txt
rm -rf tmp/


### 2.2. Перенаправления и пайпы


In [ ]:
cmd > out.txt        # перезаписать файл
cmd >> log.txt       # дозаписать в конец
cmd 2> errors.txt    # stderr в файл
cmd &> all.txt       # stdout + stderr в один файл

cmd1 | cmd2          # пайп
cmd1 | cmd2 | cmd3   # конвейер


## 3. Текстовые утилиты

### 3.1. Быстрый просмотр

- `cat`, `less`, `head`, `tail`


In [ ]:
cat file.txt
head -n 20 file.txt
tail -n 50 file.txt
tail -f app.log     # следить за логом в реальном времени


### 3.2. `grep` — поиск по тексту


In [ ]:
grep "ERROR" app.log
grep -n "ERROR" app.log            # с номерами строк
grep -R "TODO" .                   # рекурсивный поиск
grep -Ri "pattern" .               # без учёта регистра


### 3.3. `cut`, `tr`, `sort`, `uniq`, `wc`


In [ ]:
# cut: вырезать колонки
cut -d',' -f2 data.csv     # вторая колонка по разделителю ','

# tr: замена символов
echo "abc" | tr 'a-z' 'A-Z'

# sort + uniq: частоты
sort words.txt | uniq -c | sort -nr

# wc: статистика
wc -l file.txt             # количество строк
wc -w file.txt             # количество слов


### 3.4. `sed` — линейные преобразования


In [ ]:
# замена первого вхождения в каждой строке
sed 's/foo/bar/' file.txt

# глобальная замена
sed 's/foo/bar/g' file.txt

# редактирование файла "на месте" (осторожно!)
sed -i 's/foo/bar/g' file.txt


### 3.5. `awk` — мини-язык обработки таблиц и логов


In [ ]:
# вывести вторую колонку
awk '{ print $2 }' data.txt

# отфильтровать строки по числовому условию (3-я колонка > 10)
awk '$3 > 10' data.txt

# суммировать значения из 2-й колонки
awk '{ sum += $2 } END { print sum }' data.txt


## 4. Поиск в дереве файлов: `find` и друзья

### 4.1. Базовые паттерны `find`


In [ ]:
# найти все .log файлы
find . -type f -name "*.log"

# файлы больше 10 MB
find . -type f -size +10M

# файлы, изменённые за последние 24 часа
find . -type f -mtime -1

# удалить все *.tmp
find . -type f -name "*.tmp" -delete

# найти файлы и выполнить команду над каждым
find . -type f -name "*.log" -exec grep -H "ERROR" {} \;


### 4.2. `grep -R` vs `find + grep`

- `grep -R "pattern" .` — удобно для простого рекурсивного поиска
- `find` — когда нужно сложнее фильтровать по маскам, размеру, времени и т.п.


## 5. Интеграция и обработка структурированных данных

### 5.1. JSON: `jq`


In [ ]:
# вывести поле .items[0].name из JSON
jq '.items[0].name' data.json

# отфильтровать элементы по условию
jq '.items[] | select(.active == true)' data.json


### 5.2. Параллельные пайплайны: `parallel`


In [ ]:
# конвертация картинок в несколько потоков
ls *.jpg | parallel convert {} {.}.png


### 5.3. `xargs` и `tee`

- `xargs` — превращает stdin в аргументы команд
- `tee` — дублирует поток: на экран и в файл


In [ ]:
# удалить файлы, перечисленные в списке
cat files_to_remove.txt | xargs rm

# логировать вывод и одновременно видеть его в консоли
some_command | tee -a log.txt


## 6. Надёжный Bash: best practices

### 6.1. Жёсткий режим


In [ ]:
# в начале скрипта:
set -euo pipefail

# -e : выход при ошибке команды
# -u : использование неинициализированной переменной — ошибка
# -o pipefail : пайплайн считается неуспешным, если любая команда внутри упала


### 6.2. Логирование и трассировка


In [ ]:
log() {
  echo "[$(date +%F' '%T)] $*" >&2
}

log "script started"

# трассировка исполняемых команд
set -x   # включить
# ... команды ...
set +x   # выключить


## 7. Паттерны анализа логов

Допустим, строки лога выглядят так:

```text
[2024-01-05 12:10:33] ERROR something happened
```

Частый паттерн:

1. `find` — пройтись по дереву файлов
2. `grep` — отфильтровать строки по уровню (`ERROR`, `WARN`)
3. `awk` — извлечь дату/поля и агрегировать


In [ ]:
# пример: подсчёт количества ошибок по датам
find . -type f -name "*.log" -print0   | xargs -0 grep "ERROR"   | awk '
    {
      # допущение: дата всегда в начале строки в квадратных скобках
      # извлекаем всё между [ и пробелом после даты
      if (match($0, /\[[0-9-]+/)) {
        date = substr($0, RSTART+1, RLENGTH-1)
        count[date]++
      }
    }
    END {
      for (d in count) {
        printf "%s %d\n", d, count[d]
      }
    }
  ' | sort


## 8. Итог

Этот ноутбук можно использовать как:

- **шпаргалку** по синтаксису Bash
- набор **часто используемых команд и паттернов**
- стартовую точку для экспериментов — изменяй команды в ячейках и запускай их в своей среде.

Рекомендуемый следующий шаг:

- написать небольшой скрипт для своей реальной задачи (анализ логов, cleanup, деплой)
- постепенно обогащать его приёмами из этого ноутбука.
